In [3]:
# Cell 1: Install required libraries
!pip install langchain langchain-community langchain-groq -q

In [4]:
# Cell 1b: Verify installation
import langchain
import langchain_community
import langchain_groq

print("langchain:", langchain.__version__)
print("langchain_community:", langchain_community.__version__)
print("langchain_groq:", langchain_groq.__version__)

langchain: 1.2.10
langchain_community: 0.4.1
langchain_groq: 1.1.2


In [5]:
# Cell 2: Load Groq API Key from Kaggle Secrets
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
groq_api_key = user_secrets.get_secret("GROQ_API_KEY")

print("API Key loaded successfully! ✅")

API Key loaded successfully! ✅


In [6]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [7]:
import urllib.request

url = "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"
urllib.request.urlretrieve(url, "/kaggle/working/chinook.db")

print("Downloaded! ✅")

Downloaded! ✅


In [8]:
# Cell 3: Load Chinook SQLite Database
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:////kaggle/working/chinook.db")

print("Database connected! ✅")
print("Tables:", db.get_usable_table_names())

Database connected! ✅
Tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [9]:
# Cell 4: Set up Groq LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key,
    temperature=0
)

print("LLM ready! ✅")

LLM ready! ✅


In [10]:
!pip install langchain-experimental -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 4.7 MB/s eta 0:00:00a 0:00:01


In [11]:
# Cell 5: Create Text-to-SQL chain
from langchain_experimental.sql import SQLDatabaseChain

chain = SQLDatabaseChain.from_llm(llm, db, verbose=True)

print("Chain ready! ✅")

Chain ready! ✅


In [12]:
# Cell 6: Test with a question
chain.run("How many artists are there in the database?")



> Entering new SQLDatabaseChain chain...
How many artists are there in the database?
SQLQuery:

/tmp/ipykernel_55/1528193060.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  chain.run("How many artists are there in the database?")


Question: How many artists are there in the database?
SQLQuery: SELECT COUNT("ArtistId") FROM "Artist"
SQLResult: [(275,)]
Answer:Question: How many artists are there in the database?
SQLQuery: SELECT COUNT("ArtistId") FROM "Artist"
> Finished chain.


'Question: How many artists are there in the database?\nSQLQuery: SELECT COUNT("ArtistId") FROM "Artist"'

In [13]:
# Cell 7: Final clean version
def ask(question):
    result = chain.invoke({"query": question})
    raw = result['result']
    if 'SQLQuery:' in raw:
        sql = raw.split('SQLQuery:')[-1].strip()
        answer = db.run(sql)
    else:
        answer = raw
    # Clean tuple formatting
    if isinstance(answer, str):
        answer = answer.strip("[](),'")
    print(f"❓ Question: {question}")
    print(f"✅ Answer: {answer}")
    print("-" * 50)

ask("How many artists are there in the database?")
ask("How many customers are from Brazil?")
ask("What is the total number of tracks?")
ask("What is the most expensive track?")



> Entering new SQLDatabaseChain chain...
How many artists are there in the database?
SQLQuery:Question: How many artists are there in the database?
SQLQuery: SELECT COUNT("ArtistId") FROM "Artist"
SQLResult: [(275,)]
Answer:Question: How many artists are there in the database?
SQLQuery: SELECT COUNT("ArtistId") FROM "Artist"
> Finished chain.
❓ Question: How many artists are there in the database?
✅ Answer: 275
--------------------------------------------------


> Entering new SQLDatabaseChain chain...
How many customers are from Brazil?
SQLQuery:Question: How many customers are from Brazil?
SQLQuery: SELECT COUNT("CustomerId") FROM "Customer" WHERE "Country" = 'Brazil' LIMIT 5
SQLResult: [(5,)]
Answer:Question: How many customers are from Brazil?
SQLQuery: SELECT COUNT("CustomerId") FROM "Customer" WHERE "Country" = 'Brazil' LIMIT 5
> Finished chain.
❓ Question: How many customers are from Brazil?
✅ Answer: 5
--------------------------------------------------


> Entering new SQLDa